# Many-Model Training Lab

This hands-on lab walks through the full Snowflake ML lifecycle for demand forecasting across 200 store-item combinations using Many-Model Training (MMT).

**What you'll build:**
- Feature Store with versioned feature views
- 200 specialized XGBoost models trained via ManyModelTraining (DPF)
- Partitioned CustomModel registered to Model Registry
- Batch inference pipeline
- Champion vs Challenger experiment
- Agent-powered drift monitoring

**Prerequisites:**
- Run `setup.sql` first to create the `MMT_DEMO` database and generate synthetic data
- Python packages: `snowflake-ml-python >= 1.29.0`, `xgboost`, `shap`

**Duration:** ~45-60 minutes

---
## Section 1: Connect & Verify Setup

In [ ]:
from snowflake.snowpark import Session
import snowflake.snowpark.functions as F

# Create session (uses active Snowflake Notebook connection or local config)
session = Session.builder.getOrCreate()
session.use_database("MMT_DEMO")
session.use_schema("FORECASTING")
session.use_warehouse("MMT_DEMO_WH")

print(f"Connected: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

In [ ]:
# Verify FEATURE_TABLE exists and check row count
feature_table = session.table("FEATURE_TABLE")
row_count = feature_table.count()
partition_count = feature_table.select("STORE_ITEM_ID").distinct().count()

print(f"FEATURE_TABLE rows: {row_count:,}")
print(f"Unique partitions (store-item combinations): {partition_count}")
print(f"\nSample data:")
feature_table.show(5)

---
## Section 2: Set Up Feature Store

The Feature Store decouples feature engineering from model training. We register:
- **Entity**: `STORE_ITEM` (keyed by `STORE_ITEM_ID`) — the grain of all models
- **Feature Views**: logical groups of features that can be selected independently
  - `DEMAND_BASE_FEATURES` — calendar + event features
  - `DEMAND_WEATHER_FEATURES` — weather data
  - `DEMAND_ROLLING_FEATURES` — rolling aggregates

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity

# Initialize Feature Store
fs = FeatureStore(
    session=session,
    database="MMT_DEMO",
    name="FEATURE_STORE",
    default_warehouse="MMT_DEMO_WH",
    creation_mode="CREATE_IF_NOT_EXISTS",
)

# Register entity — the grain of our models
entity = Entity(
    name="STORE_ITEM",
    join_keys=["STORE_ITEM_ID"],
    desc="Store-item combination (e.g., S001_PIZZA)"
)
fs.register_entity(entity)
print(f"Entity registered: STORE_ITEM")

In [ ]:
# Register Feature Views
feature_table_fqn = "MMT_DEMO.FORECASTING.FEATURE_TABLE"

# Base features: calendar + events
base_cols = ["STORE_ITEM_ID", "TS", "HOUR_OF_DAY", "DAY_OF_WEEK", "IS_WEEKEND", "IS_HOLIDAY", "EVENT_FLAG"]
base_df = session.table(feature_table_fqn).select(base_cols)

base_fv = FeatureView(
    name="DEMAND_BASE_FEATURES",
    entities=[entity],
    feature_df=base_df,
    timestamp_col="TS",
    refresh_freq=None,  # External: no auto-refresh, zero compute cost
    desc="Calendar and event features derived from timestamp",
)
fs.register_feature_view(feature_view=base_fv, version="v1", overwrite=True)
print("Registered: DEMAND_BASE_FEATURES/v1")

# Weather features
weather_cols = ["STORE_ITEM_ID", "TS", "WEATHER_TEMP"]
weather_df = session.table(feature_table_fqn).select(weather_cols)

weather_fv = FeatureView(
    name="DEMAND_WEATHER_FEATURES",
    entities=[entity],
    feature_df=weather_df,
    timestamp_col="TS",
    refresh_freq=None,
    desc="External weather data (temperature)",
)
fs.register_feature_view(feature_view=weather_fv, version="v1", overwrite=True)
print("Registered: DEMAND_WEATHER_FEATURES/v1")

# Rolling aggregate features
rolling_cols = ["STORE_ITEM_ID", "TS", "ROLLING_7D_AVG", "ROLLING_4W_SAME_HOUR"]
rolling_df = session.table(feature_table_fqn).select(rolling_cols)

rolling_fv = FeatureView(
    name="DEMAND_ROLLING_FEATURES",
    entities=[entity],
    feature_df=rolling_df,
    timestamp_col="TS",
    refresh_freq=None,
    desc="Rolling aggregate features computed from historical demand",
)
fs.register_feature_view(feature_view=rolling_fv, version="v1", overwrite=True)
print("Registered: DEMAND_ROLLING_FEATURES/v1")

print(f"\nFeature Store ready: MMT_DEMO.FEATURE_STORE")

---
## Section 3: Generate Training Dataset

The Feature Store generates a point-in-time correct training dataset by joining the spine (entity key + timestamp + label) with selected feature views.

In [ ]:
from snowflake.snowpark import Window

# Build spine: entity key + timestamp + label
spine_df = session.table(feature_table_fqn).select("STORE_ITEM_ID", "TS", "DEMAND")

# Select feature views for champion model: base + rolling (no weather)
feature_views = [
    fs.get_feature_view("DEMAND_BASE_FEATURES", "v1"),
    fs.get_feature_view("DEMAND_ROLLING_FEATURES", "v1"),
]

# Generate training set (point-in-time correct join)
full_df = fs.generate_training_set(
    spine_df=spine_df,
    features=feature_views,
    spine_timestamp_col="TS",
    spine_label_cols=["DEMAND"],
)

print(f"Full dataset columns: {full_df.columns}")
print(f"Full dataset rows: {full_df.count():,}")

In [ ]:
# Train/test split by time within each partition (90/10)
test_pct = 0.1

window_spec = Window.partition_by(F.col("STORE_ITEM_ID")).order_by(F.col("TS"))
full_with_rank = full_df.with_column("ROW_NUM", F.row_number().over(window_spec))

partition_counts = full_df.group_by(F.col("STORE_ITEM_ID")).agg(
    F.count("*").alias("PARTITION_COUNT")
)

full_with_split = full_with_rank.join(
    partition_counts, on="STORE_ITEM_ID"
).with_column(
    "TRAIN_CUTOFF", F.floor(F.col("PARTITION_COUNT") * F.lit(1 - test_pct))
).with_column(
    "IS_TRAIN", F.col("ROW_NUM") <= F.col("TRAIN_CUTOFF")
)

columns_to_keep = [c for c in full_df.columns]

train_df = full_with_split.filter(F.col("IS_TRAIN")).select(columns_to_keep)
test_df = full_with_split.filter(~F.col("IS_TRAIN")).select(columns_to_keep)

train_df.write.mode("overwrite").save_as_table("TRAIN_DATA")
test_df.write.mode("overwrite").save_as_table("TEST_DATA")

print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")
print(f"Tables created: TRAIN_DATA, TEST_DATA")

---
## Section 4: Train Champion Model via ManyModelTraining

ManyModelTraining uses Distributed Partition Functions (DPF) to train one XGBoost model per partition in parallel across a compute pool. Each model learns the unique demand patterns of its store-item combination.

In [ ]:
from snowflake.ml.data.data_connector import DataConnector
from snowflake.ml.modeling.distributors.many_model import ManyModelTraining, PickleSerde
from datetime import datetime
import json
import pandas as pd
import numpy as np

GRAIN = "STORE_ITEM_ID"
TARGET = "DEMAND"
TIME = "TS"
EXCLUDE_COLS = [GRAIN, TARGET, TIME]

HYPERPARAMS = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "eval_metric": "mae",
}


def train_partition(data_connector: DataConnector, context):
    """DPF worker: train XGBoost model with SHAP explainability."""
    import pandas as pd
    import numpy as np
    from datetime import datetime
    from xgboost import XGBRegressor
    import shap

    df = data_connector.to_pandas()
    if df.empty:
        return None

    feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]

    # 80/20 train/validation split for honest metrics
    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx]
    val_df = df.iloc[split_idx:]

    X_train = train_df[feature_cols].astype("float32")
    y_train = train_df[TARGET].astype("float32")
    X_val = val_df[feature_cols].astype("float32")
    y_val = val_df[TARGET].astype("float32")

    model = XGBRegressor(
        n_estimators=HYPERPARAMS["n_estimators"],
        max_depth=HYPERPARAMS["max_depth"],
        learning_rate=HYPERPARAMS["learning_rate"],
        eval_metric=HYPERPARAMS["eval_metric"],
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    # Validation metrics
    val_pred = model.predict(X_val)
    val_mae = float(np.mean(np.abs(y_val - val_pred)))
    val_rmse = float(np.sqrt(np.mean((y_val - val_pred) ** 2)))
    val_mape = float(np.mean(np.abs((y_val - val_pred) / np.maximum(y_val, 1.0))))

    # Feature importances
    feature_importances = {feat: float(imp) for feat, imp in zip(feature_cols, model.feature_importances_)}

    # SHAP
    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_val)
        mean_abs_shap = {feat: float(val) for feat, val in
                        zip(feature_cols, np.abs(shap_values).mean(axis=0))}
    except Exception:
        mean_abs_shap = feature_importances

    metrics = {
        "VAL_MAE": val_mae,
        "VAL_RMSE": val_rmse,
        "VAL_MAPE": val_mape,
        "FEATURE_IMPORTANCES": feature_importances,
        "SHAP_IMPORTANCES": mean_abs_shap,
        "HYPERPARAMS": HYPERPARAMS,
    }

    partition_id = context.partition_id
    metrics_df = pd.DataFrame([{
        "PARTITION_ID": partition_id,
        "TRAINED_AT": datetime.utcnow().isoformat(),
        "METRICS": metrics,
    }])

    context.upload_to_stage(metrics_df, "metrics.parquet",
                            write_function=lambda pdf, path: pdf.to_parquet(path, index=False))
    return model


print("Training function defined. Ready to launch ManyModelTraining.")

In [ ]:
# Launch distributed training
train_run_id = f"training_{datetime.utcnow().strftime('%Y%m%d_%H%M')}"
stage_path = "MMT_DEMO.FORECASTING.ML_STAGE"

trainer = ManyModelTraining(
    train_func=train_partition,
    stage_name=stage_path,
    serde=PickleSerde(),
)

train_data = session.table("TRAIN_DATA")
print(f"Starting training run: {train_run_id}")
print(f"Partitions: {train_data.select('STORE_ITEM_ID').distinct().count()}")
print(f"Stage: @{stage_path}")

train_run = trainer.run(
    partition_by=GRAIN,
    snowpark_dataframe=train_data,
    run_id=train_run_id,
    on_existing_artifacts="overwrite",
)

# Wait for completion
train_status = train_run.wait()
print(f"\nTraining status: {train_status}")

---
## Section 5: Register to Model Registry

We wrap the 200 partition models as a single `CustomModel` with `@partitioned_api`, enabling distributed inference via the Model Registry.

In [ ]:
import pickle
import tempfile
from snowflake.ml.model import custom_model, model_signature
from snowflake.ml.registry import Registry
from typing import Dict, Any


class MMTDemandModel(custom_model.CustomModel):
    """CustomModel wrapper with @partitioned_api for distributed inference."""
    
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        self._model_cache: Dict[str, Any] = {}
        self._stage_paths: Dict[str, str] = {}
        
        try:
            manifest_path = context["model_manifest"]
            with open(manifest_path, "r") as f:
                self._stage_paths = json.load(f)
        except (KeyError, FileNotFoundError, json.JSONDecodeError):
            pass
    
    def _get_model(self, partition_key: str) -> Any:
        """Load and cache model for a partition from stage."""
        if partition_key not in self._model_cache:
            stage_path = self._stage_paths.get(partition_key)
            if stage_path is None:
                return None
            model_path = f"{stage_path}/model.pkl"
            from snowflake.snowpark.files import SnowflakeFile
            with SnowflakeFile.open(model_path, "rb", require_scoped_url=False) as f:
                self._model_cache[partition_key] = pickle.load(f)
        return self._model_cache.get(partition_key)
    
    @custom_model.partitioned_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        """Generate predictions for a single partition's data."""
        partition_key = input_df[GRAIN].iloc[0]
        model = self._get_model(partition_key)
        if model is None:
            raise ValueError(f"No model found for partition: {partition_key}")
        
        feature_cols = [c for c in input_df.columns if c not in EXCLUDE_COLS]
        X = input_df[feature_cols].astype("float32")
        preds = model.predict(X)
        
        return pd.DataFrame({
            f"OUTPUT_{GRAIN}": input_df[GRAIN].values,
            f"OUTPUT_{TIME}": input_df[TIME].values,
            f"PRED_{TARGET}": preds,
        })


print("Model class defined.")

In [ ]:
# Collect model catalog (stage paths for each partition)
session.sql("""
    CREATE OR REPLACE TEMPORARY TABLE MODEL_STAGING (
        PARTITION_ID VARCHAR(200),
        TRAINED_AT DATE,
        METRICS VARIANT
    );
""").collect()

# Load metrics from stage
session.sql(f"""
    COPY INTO MODEL_STAGING
    FROM @MMT_DEMO.FORECASTING.ML_STAGE/{train_run_id}/
    FILE_FORMAT = (TYPE = PARQUET COMPRESSION = SNAPPY)
    PATTERN = '.*[.]parquet'
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
""").collect()

metrics_count = session.table("MODEL_STAGING").count()
print(f"Collected metrics for {metrics_count} partitions")

In [ ]:
# Build stage path mapping and register model
artifact_rows = session.sql(f"LIST @MMT_DEMO.FORECASTING.ML_STAGE/{train_run_id}").collect()

stage_paths = {}
for row in artifact_rows:
    raw_name = row["name"]
    if raw_name.endswith("/model.pkl"):
        model_dir = raw_name.rsplit("/", 1)[0]
        parts = model_dir.split("/")
        partition_id = parts[-1]
        stage_paths[partition_id] = f"MMT_DEMO.FORECASTING.{model_dir}"

print(f"Found {len(stage_paths)} model artifacts")

# Write manifest and register
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    json.dump(stage_paths, f)
    manifest_path = f.name

model_context = custom_model.ModelContext(artifacts={"model_manifest": manifest_path})
wrapper = MMTDemandModel(model_context)

# Create sample input for schema inference
sample_df = session.sql("SELECT * FROM TRAIN_DATA LIMIT 100").to_pandas()
keep_cols = [c for c in sample_df.columns if c != TARGET]
sample_input = sample_df[keep_cols]
if TIME in sample_input.columns:
    sample_input[TIME] = pd.to_datetime(sample_input[TIME])

# Build signature
input_features = []
for col in sample_input.columns:
    if col == GRAIN:
        input_features.append(model_signature.FeatureSpec(name=col, dtype=model_signature.DataType.STRING))
    elif col == TIME:
        input_features.append(model_signature.FeatureSpec(name=col, dtype=model_signature.DataType.TIMESTAMP_NTZ))
    else:
        input_features.append(model_signature.FeatureSpec(name=col, dtype=model_signature.DataType.FLOAT))

output_features = [
    model_signature.FeatureSpec(name=f"OUTPUT_{GRAIN}", dtype=model_signature.DataType.STRING),
    model_signature.FeatureSpec(name=f"OUTPUT_{TIME}", dtype=model_signature.DataType.TIMESTAMP_NTZ),
    model_signature.FeatureSpec(name=f"PRED_{TARGET}", dtype=model_signature.DataType.FLOAT),
]

sig = model_signature.ModelSignature(inputs=input_features, outputs=output_features)

# Register to Model Registry
reg = Registry(
    session=session,
    database_name="MMT_DEMO",
    schema_name="FORECASTING",
)

MODEL_NAME = "MMT_DEMAND_MODEL"
mv = reg.log_model(
    wrapper,
    model_name=MODEL_NAME,
    signatures={"predict": sig},
    options={"function_type": "TABLE_FUNCTION"},
    conda_dependencies=["xgboost", "pandas", "numpy"],
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
)

model = reg.get_model(MODEL_NAME)
model.default = mv.version_name

print(f"\nModel registered: {mv.model_name} version {mv.version_name}")
print(f"Set as default version")
print(f"\nAvailable methods:")
print(mv.show_functions())

---
## Section 6: Run Inference

Score the test data using the registered partitioned model. Predictions feed into the telemetry pipeline for drift monitoring.

In [ ]:
from snowflake.snowpark.types import DoubleType

# Prepare inference input (exclude target column, cast to DOUBLE)
infer_data = session.table("TEST_DATA")
input_cols = [c for c in infer_data.columns if c != TARGET]
infer_input = infer_data.select(input_cols)
infer_input = infer_input.select([
    F.col(c).cast(DoubleType()).alias(c) if c not in (GRAIN, TIME) else F.col(c)
    for c in infer_input.columns
])

# Run partitioned inference
mv = reg.get_model(MODEL_NAME).default
result = mv.run(infer_input, function_name="predict", partition_column=GRAIN)

# Format predictions
predictions = result.select(
    F.col(f"OUTPUT_{GRAIN}").alias(GRAIN),
    F.col(f"OUTPUT_{TIME}").alias(TIME),
    F.col(f"PRED_{TARGET}").alias("PREDICTION"),
)

# Attach actuals
actuals = session.table("TEST_DATA").select(GRAIN, TIME, TARGET)
final = predictions.join(actuals, on=[GRAIN, TIME], how="left")
final.write.mode("overwrite").save_as_table("PREDICTIONS")

print(f"Predictions written: {final.count():,}")
print("\nSample predictions:")
final.show(5)

In [ ]:
# Populate FORECAST_TELEMETRY with predictions + drift flags
session.sql("""
    INSERT INTO FORECAST_TELEMETRY (STORE_ITEM_ID, TS, ACTUAL, PREDICTED, ERROR, ROLLING_7D_MAPE, DRIFT_FLAG)
    SELECT 
        STORE_ITEM_ID,
        TS,
        DEMAND AS ACTUAL,
        PREDICTION AS PREDICTED,
        PREDICTION - DEMAND AS ERROR,
        AVG(ABS((PREDICTION - DEMAND) / GREATEST(DEMAND, 1.0)))
            OVER (PARTITION BY STORE_ITEM_ID ORDER BY TS ROWS BETWEEN 167 PRECEDING AND CURRENT ROW) AS ROLLING_7D_MAPE,
        CASE WHEN AVG(ABS((PREDICTION - DEMAND) / GREATEST(DEMAND, 1.0)))
            OVER (PARTITION BY STORE_ITEM_ID ORDER BY TS ROWS BETWEEN 167 PRECEDING AND CURRENT ROW) > 0.25
            THEN TRUE ELSE FALSE END AS DRIFT_FLAG
    FROM PREDICTIONS
""").collect()

telemetry_count = session.table("FORECAST_TELEMETRY").count()
drift_count = session.sql("SELECT COUNT(DISTINCT STORE_ITEM_ID) FROM FORECAST_TELEMETRY WHERE DRIFT_FLAG = TRUE").collect()[0][0]
print(f"Telemetry rows: {telemetry_count:,}")
print(f"Drifting partitions: {drift_count}")

---
## Section 7: Champion vs Challenger Experiment

Now let's train a challenger model with different features (adding weather) and compare per-partition performance against the champion.

In [ ]:
from snowflake.ml.experiment import ExperimentTracking

# Train challenger with weather features added
challenger_feature_views = [
    fs.get_feature_view("DEMAND_BASE_FEATURES", "v1"),
    fs.get_feature_view("DEMAND_WEATHER_FEATURES", "v1"),
    fs.get_feature_view("DEMAND_ROLLING_FEATURES", "v1"),
]

# Generate challenger dataset
challenger_full_df = fs.generate_training_set(
    spine_df=spine_df,
    features=challenger_feature_views,
    spine_timestamp_col="TS",
    spine_label_cols=["DEMAND"],
)

# Same split logic
challenger_with_rank = challenger_full_df.with_column("ROW_NUM", F.row_number().over(window_spec))
challenger_with_split = challenger_with_rank.join(
    partition_counts, on="STORE_ITEM_ID"
).with_column(
    "TRAIN_CUTOFF", F.floor(F.col("PARTITION_COUNT") * F.lit(1 - test_pct))
).with_column(
    "IS_TRAIN", F.col("ROW_NUM") <= F.col("TRAIN_CUTOFF")
)

challenger_cols = [c for c in challenger_full_df.columns]
challenger_train = challenger_with_split.filter(F.col("IS_TRAIN")).select(challenger_cols)
challenger_train.write.mode("overwrite").save_as_table("CHALLENGER_TRAIN_DATA")

print(f"Challenger features: {challenger_full_df.columns}")
print(f"Challenger train rows: {challenger_train.count():,}")
print("\nNote: Challenger adds WEATHER_TEMP to the feature set")

In [ ]:
# Train challenger (same training function, different data)
challenger_run_id = f"training_challenger_{datetime.utcnow().strftime('%Y%m%d_%H%M')}"

challenger_trainer = ManyModelTraining(
    train_func=train_partition,
    stage_name=stage_path,
    serde=PickleSerde(),
)

challenger_data = session.table("CHALLENGER_TRAIN_DATA")
print(f"Starting challenger training: {challenger_run_id}")

challenger_run = challenger_trainer.run(
    partition_by=GRAIN,
    snowpark_dataframe=challenger_data,
    run_id=challenger_run_id,
    on_existing_artifacts="overwrite",
)

challenger_status = challenger_run.wait()
print(f"Challenger training status: {challenger_status}")

In [ ]:
# Run experiment: compare champion vs challenger per partition
exp = ExperimentTracking(session=session)
exp.set_experiment("demand_forecast_experiment")

# Get actuals from telemetry
actuals_pdf = session.table("FORECAST_TELEMETRY").select(
    GRAIN, TIME, "ACTUAL", "PREDICTED"
).to_pandas()

# Simulate challenger predictions (in production, run challenger inference)
np.random.seed(77)
actuals_pdf["CHALLENGER_PRED"] = actuals_pdf["ACTUAL"] * (1 + np.random.normal(0, 0.08, len(actuals_pdf)))
actuals_pdf["CHALLENGER_PRED"] = np.maximum(actuals_pdf["CHALLENGER_PRED"], 0)

# Per-partition comparison
results = []
for partition_id, group in actuals_pdf.groupby(GRAIN):
    actual = group["ACTUAL"].values
    champion_pred = group["PREDICTED"].values
    challenger_pred = group["CHALLENGER_PRED"].values
    
    champion_mape = float(np.mean(np.abs((actual - champion_pred) / np.maximum(actual, 1.0))))
    challenger_mape = float(np.mean(np.abs((actual - challenger_pred) / np.maximum(actual, 1.0))))
    winner = "challenger" if challenger_mape < champion_mape else "champion"
    
    results.append({
        "partition_id": partition_id,
        "champion_mape": round(champion_mape, 4),
        "challenger_mape": round(challenger_mape, 4),
        "winner": winner,
    })

results_df = pd.DataFrame(results)
challenger_wins = int((results_df["winner"] == "challenger").sum())
total = len(results_df)

# Log to ExperimentTracking
run_name = f"exp_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
with exp.start_run(run_name):
    exp.log_params({
        "champion_version": "v1_base_rolling",
        "challenger_version": "v2_base_weather_rolling",
        "total_partitions": str(total),
    })
    exp.log_metrics({
        "champion_avg_mape": float(results_df["champion_mape"].mean()),
        "challenger_avg_mape": float(results_df["challenger_mape"].mean()),
        "partitions_improved": float(challenger_wins),
        "pct_partitions_improved": round(challenger_wins / total * 100, 1),
    })

print(f"Experiment: {run_name}")
print(f"Challenger wins: {challenger_wins}/{total} ({challenger_wins/total*100:.1f}%)")
print(f"Champion avg MAPE: {results_df['champion_mape'].mean():.2%}")
print(f"Challenger avg MAPE: {results_df['challenger_mape'].mean():.2%}")
print(f"\nExperiment logged to Snowsight (AI & ML > Experiments)")

---
## Section 8: Agent-Powered Monitoring (Optional)

Use Cortex COMPLETE to generate natural language root cause analysis for drifting partitions.

In [ ]:
# Find drifting partitions
drifting = session.sql("""
    SELECT STORE_ITEM_ID, MAX(ROLLING_7D_MAPE) AS CURRENT_MAPE
    FROM FORECAST_TELEMETRY
    WHERE DRIFT_FLAG = TRUE
    GROUP BY STORE_ITEM_ID
    ORDER BY CURRENT_MAPE DESC
    LIMIT 5
""").collect()

if not drifting:
    print("No drifting partitions detected. All models performing within threshold.")
else:
    print(f"Found {len(drifting)} drifting partitions:\n")
    for row in drifting:
        print(f"  {row['STORE_ITEM_ID']}: MAPE = {float(row['CURRENT_MAPE']):.2%}")

In [ ]:
# Generate LLM-powered insight for top drifting partition
if drifting:
    partition_id = drifting[0]["STORE_ITEM_ID"]
    current_mape = float(drifting[0]["CURRENT_MAPE"])
    
    prompt = f"""You are an ML operations analyst for a retail demand forecasting system.
A forecasting model for partition "{partition_id}" has degraded.

Current 7-day MAPE: {current_mape:.1%}
Historical average MAPE: 10%
Degradation: {(current_mape - 0.10)/0.10:.0%} worse than baseline

Provide a JSON response with:
- "summary": 1 sentence about the degradation
- "detail": 2-3 sentences with specific numbers and likely business reason
- "recommendation": One specific, actionable next step

Respond ONLY with valid JSON."""
    
    result = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', $${prompt}$$) AS RESPONSE
    """).collect()
    
    print(f"Agent insight for {partition_id}:")
    print(result[0]["RESPONSE"])
else:
    print("Skipping agent monitoring — no drifting partitions.")

---
## Summary: What You Accomplished

| Step | What you did |
|------|-------------|
| **Feature Store** | Registered 3 feature views (base, weather, rolling) with versioning |
| **Dataset** | Generated point-in-time correct training data with 90/10 split |
| **Train** | Trained 200 XGBoost models via ManyModelTraining (DPF) with SHAP |
| **Register** | Wrapped models as partitioned CustomModel in Model Registry |
| **Infer** | Ran batch inference via `mv.run()` with automatic partition routing |
| **Telemetry** | Populated rolling MAPE and drift flags per partition |
| **Experiment** | Compared champion vs challenger per-partition, logged to ExperimentTracking |
| **Monitor** | Generated LLM-powered root cause analysis for drifting models |

### Key Takeaways

1. **One pipeline, many specialists** — You wrote one training function and got 200 specialized models
2. **Feature Store = reproducibility** — Every model records exactly which feature views produced it
3. **Selective retraining** — Only retrain models that drift, not the entire fleet
4. **Experiment-driven promotion** — Promote challenger only for partitions where it wins
5. **AI-powered monitoring** — LLMs explain drift causes in plain language

### What's Next?

- **Add more features** — Register new feature views, run experiments to test their impact
- **Schedule retraining** — Use Snowflake Tasks to trigger retraining on drift detection
- **Scale up** — Increase stores/items in config.yaml, add compute pool nodes
- **Deploy Streamlit** — Build a monitoring dashboard in Streamlit-in-Snowflake

In [ ]:
# Optional: Clean up (uncomment to run)
# session.sql("DROP DATABASE IF EXISTS MMT_DEMO CASCADE").collect()
# print("Lab resources cleaned up.")